# CertifyTube Viva Notebook - Quiz Verification Layer

This notebook demonstrates the second verification layer end-to-end in one place.
It includes:
1. Contract preview (`/quiz/generate`)
2. Stable mocked API demo (works without OpenRouter key)
3. Optional live provider call

In [ ]:
# If running in a fresh Colab runtime, uncomment and run:
# !git clone https://github.com/your-username/certifytube_ml_model.git
# %cd certifytube_ml_model

from pathlib import Path
ROOT = Path.cwd()
print('Working directory:', ROOT)
assert (ROOT / 'app').exists(), 'Run this notebook from project root.'

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from app.api.quiz_schemas import GenerateQuizRequest, GenerateQuizResponse

print('Request schema fields:')
print(GenerateQuizRequest.model_json_schema()['properties'].keys())
print('
Response schema fields:')
print(GenerateQuizResponse.model_json_schema()['properties'].keys())

In [ ]:
# Stable viva demo: monkeypatch quiz route dependencies to avoid external API/network dependency.
from fastapi.testclient import TestClient
import app.api.quiz_routes as quiz_routes
from verification.quiz.transcript.processor import ProcessedTranscript, TranscriptChunk
from app.main import app


def fake_fetch_and_process_transcript(video_id_or_url: str) -> ProcessedTranscript:
    text = 'Gradient descent updates model weights in the opposite direction of the gradient.'
    chunk = TranscriptChunk(chunk_id='chunk_1', text=text, token_estimate=25, has_code=False)
    return ProcessedTranscript(chunks=[chunk], has_code_content=False)


def fake_generate_quiz(session_id, video_title, transcript, video_duration_sec, include_coding):
    return [
        {
            'question_id': 'q1',
            'type': 'mcq',
            'question': 'What controls the step size in gradient descent?',
            'options': ['Learning rate', 'Batch size', 'Epoch count', 'Dropout'],
            'correct_answer': 'Learning rate',
            'explanation': 'The transcript explains that learning rate controls update size.',
            'difficulty': 'easy',
        }
    ]

orig_fetch = quiz_routes.fetch_and_process_transcript
orig_generate = quiz_routes.generate_quiz
quiz_routes.fetch_and_process_transcript = fake_fetch_and_process_transcript
quiz_routes.generate_quiz = fake_generate_quiz

try:
    client = TestClient(app)
    payload = {
        'session_id': 'viva-quiz-001',
        'video_id': 'rfscVS0vtbw',
        'video_duration_sec': 14400,
    }
    response = client.post('/quiz/generate', json=payload)
    print('Status:', response.status_code)
    result = response.json()
    print('Top-level keys:', list(result.keys()))
    print('Question type:', result['questions'][0]['type'])
    result
finally:
    quiz_routes.fetch_and_process_transcript = orig_fetch
    quiz_routes.generate_quiz = orig_generate

In [ ]:
# Optional live call (requires OPENROUTER_API_KEY and internet access).
# Keep this off during viva unless you want a live provider demonstration.

RUN_LIVE = False
if RUN_LIVE:
    from fastapi.testclient import TestClient
    from app.main import app

    client = TestClient(app)
    payload = {
        'session_id': 'viva-quiz-live-001',
        'video_id': 'rfscVS0vtbw',
        'video_duration_sec': 14400,
    }
    response = client.post('/quiz/generate', json=payload)
    print('Status:', response.status_code)
    response.json()

## Viva Speaking Points (Quiz Layer)
- Quiz is transcript-grounded and returned with correct answers + explanations.
- Backend responsibility: present questions, collect answers, compute pass/fail.
- This is the second independent verification layer after engagement scoring.